In [3]:
import os
import sys
import logging
from typing import TypedDict
from dotenv import load_dotenv
import chromadb
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from langgraph.graph import StateGraph, START, END

# Initialize strict text configuration tracking outputs via stdout
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - [%(levelname)s] - %(name)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("RAGReconciliationAgent")

# Synchronize env variables from file structure
load_dotenv()
logger.info("Cloud platform credentials and connection paths synchronized successfully.")

2026-07-21 12:05:34,483 - [INFO] - RAGReconciliationAgent - Cloud platform credentials and connection paths synchronized successfully.


In [4]:
# Initialize persistent client storage on local disk infrastructure
chroma_client = chromadb.PersistentClient(path="./chroma_financial_db")

# Provision an isolated schema bucket collection for unstructured remittance advice sheets
try:
    remittance_collection = chroma_client.get_or_create_collection(
        name="customer_remittances"
    )
    logger.info("Chroma Vector Database collection successfully initialized.")
    
    # Ingestion & Embeddings Phase: Load loose raw document snippets into text chunks
    sample_document_chunks = [
        "Invoice INV-2026-A1: Item pricing mismatch identified on line 3. Short paying five hundred dollars.",
        "Invoice INV-2026-B2: Delay encountered in freight transportation delivery. Waiving shipping fee balance of two hundred dollars.",
        "Invoice INV-2026-C3: Pallet package arrived crushed at warehouse facility. Damaged items claim deducted.",
        "Invoice INV-2026-D4: Standard application of state corporate tax exemption certificate validation.",
        "Invoice INV-2026-E5: Customer exercised the contractually approved two percent early payment discount taken fee incentive."
    ]
    
    sample_metadata = [
        {"customer": "Accenture Batch A", "invoice_id": "INV-2026-A1"},
        {"customer": "Accenture Batch B", "invoice_id": "INV-2026-B2"},
        {"customer": "Accenture Batch C", "invoice_id": "INV-2026-C3"},
        {"customer": "Accenture Batch D", "invoice_id": "INV-2026-D4"},
        {"customer": "Accenture Batch E", "invoice_id": "INV-2026-E5"}
    ]
    
    sample_chunk_ids = ["chunk_d01", "chunk_d02", "chunk_d03", "chunk_d04", "chunk_d05"]
    
    # Insert sentences; Chroma triggers default internal embeddings models to generate coordinate vectors
    remittance_collection.add(
        documents=sample_document_chunks,
        metadatas=sample_metadata,
        ids=sample_chunk_ids
    )
    logger.info(f"Ingestion pipeline complete: {len(sample_document_chunks)} remittance text chunks successfully embedded and indexed.")
    
except Exception as ex:
    logger.error(f"Vector Database population failure: {str(ex)}")
    raise ex

2026-07-21 12:05:34,510 - [INFO] - RAGReconciliationAgent - Chroma Vector Database collection successfully initialized.
2026-07-21 12:05:35,207 - [INFO] - RAGReconciliationAgent - Ingestion pipeline complete: 5 remittance text chunks successfully embedded and indexed.


In [5]:

# 1. RAG Search Retrieval Tool
def search_remittance(query_context: str) -> str:
    """Searches the Chroma vector storage index using semantic query text to isolate target explanation context chunks."""
    logger.info(f"[TOOL EXECUTION] -> search_remittance searching vector DB for context: '{query_context}'")
    
    search_results = remittance_collection.query(
        query_texts=[query_context],
        n_results=1
    )
    
    if search_results and search_results['documents']:
        matched_chunk = search_results['documents'][0][0]
        logger.info(f"[TOOL RESULT] -> Search success. Isolated relevant text chunk: '{matched_chunk}'")
        return matched_chunk
    
    logger.warning("[TOOL RESULT] -> Zero relevant reference data blocks found in database index.")
    return "No matching remittance text evidence discovered."

# 2. ERP Accounts Receivable Database Tool
def query_open_ar(customer_name: str) -> dict:
    """Queries corporate system ledgers for open invoices matching target account metrics."""
    logger.info(f"[TOOL EXECUTION] -> query_open_ar looking up open balance items for: {customer_name}")
    # Simulates active accounting record details pulled from back-office infrastructure
    return {
        "invoice_code": "INV-2026-A1",
        "expected_invoice_amount": 10000.00,
        "is_auditable": True
    }

# 3. Arithmetic Variance Calculator Tool
def calculate_variance(bank_amount: float, erp_amount: float) -> float:
    """Performs strict numerical subtraction outside the AI prompt frame to securely identify precise ledger gap values."""
    logger.info(f"[TOOL EXECUTION] -> calculate_variance parsing math values: Billed ${erp_amount:.2f}, Paid ${bank_amount:.2f}")
    return float(erp_amount - bank_amount)

# 4. Cash Application Ledger Updating Action Tool
def mark_invoice_closed(invoice_id: str) -> bool:
    """Writes an official state update directly to the internal ERP table closing out active billing fields."""
    logger.info(f"[TOOL EXECUTION] -> mark_invoice_closed updating record item {invoice_id} status indicator to CLOSED.")
    return True

# 5. Dispute File Creation Action Tool
def create_deduction_record(invoice_id: str, deduction_amount: float, code: str) -> str:
    """Generates an official open dispute record item inside the tracking registry tagged with a code."""
    logger.info(f"[TOOL EXECUTION] -> create_deduction_record logging dispute for {invoice_id}. Amount: ${deduction_amount:.2f}, Assigned Code: {code}")
    return f"DISPUTE-{invoice_id}-GENERATED"

logger.info("Complete documented function tool library successfully compiled and operational.")

2026-07-21 12:05:35,234 - [INFO] - RAGReconciliationAgent - Complete documented function tool library successfully compiled and operational.


In [ ]:
import time
from collections import namedtuple
from typing import TypedDict
from azure.ai.projects import AIProjectClient

# Create a mock token object matching the Azure core signature
AccessToken = namedtuple("AccessToken", ["token", "expires_on"])

class StaticTokenCredential:
    """Wrapper to safely route a key string into token-based Azure client initializers."""
    def __init__(self, key: str):
        self.key = key

    def get_token(self, *scopes, **kwargs):
        # Return the key as the token payload with a distant expiration timestamp
        return AccessToken(token=self.key, expires_on=int(time.time()) + 3600)


class RAGAgentState(TypedDict):
    customer_name: str
    bank_payment_value: float
    remittance_search_query: str
    graph_invoice_id: str
    graph_invoice_amount: float
    calculated_variance_amount: float
    retrieved_document_grounding: str
    assigned_dispute_reason_code: str
    system_execution_logs: list


# Authenticate local software parameters with cloud gateway platforms

try:
    # Wrap the hardcoded key into a token-compatible format expected by Azure authentication policies
    credential = StaticTokenCredential(AZURE_AI_PROJECT_KEY)
    
    azure_client = AIProjectClient(
        endpoint=AZURE_AI_PROJECT_ENDPOINT,
        credential=credential
    )

    logger.info(f"Connected securely to Azure AI Project Endpoint: {AZURE_AI_PROJECT_ENDPOINT}")
except Exception as ex:
    logger.error(f"Critical initialization failure at cloud infrastructure layer: {str(ex)}")
    raise ex

2026-07-21 12:05:35,262 - [INFO] - RAGReconciliationAgent - Connected securely to Azure AI Project Endpoint: https://hub-accenture-training.services.ai.azure.com/api/projects/project-agentic-matching


In [7]:
def check_financial_ledger_node(state: RAGAgentState):
    logger.info("--- ENTERING NODE: Financial Ledger Evaluator ---")
    logs = state.get("system_execution_logs", [])
    logs.append("Triggered initial balance tracking checks using custom tool infrastructure.")
    
    # 1. Trigger ERP database reading tool
    erp_data = query_open_ar(state["customer_name"])
    
    # 2. Trigger hardware-level arithmetic validation tool
    variance = calculate_variance(state["bank_payment_value"], erp_data["expected_invoice_amount"])
    
    logger.info(f"Math analysis result: Discrepancy evaluated to: ${variance:.2f}")
    
    return {
        "graph_invoice_id": erp_data["invoice_code"],
        "graph_invoice_amount": erp_data["expected_invoice_amount"],
        "calculated_variance_amount": variance,
        "system_execution_logs": logs
    }

def document_retrieval_rag_node(state: RAGAgentState):
    logger.info("--- ENTERING NODE: Document Retrieval RAG Processor ---")
    logs = state.get("system_execution_logs", [])
    logs.append("Triggered targeted semantic vector lookup pass over unstructured PDF data collections.")
    
    # 3. Trigger semantic retrieval tool over local database collections
    extracted_text_evidence = search_remittance(state["remittance_search_query"])
    
    return {
        "retrieved_document_grounding": extracted_text_evidence,
        "system_execution_logs": logs
    }

def response_grounding_validation_node(state: RAGAgentState):
    logger.info("--- ENTERING NODE: Response Grounding Engine ---")
    logs = state.get("system_execution_logs", [])
    logs.append("Executing grounded inference classification check against Azure cloud models.")
    
    # Implementation of strict hardcoded programmatic validations outside model control boundaries
    if state["calculated_variance_amount"] <= 5.00:
        logger.info("Calculated gap falls inside standard corporate tolerances. Auto-resolving.")
        mark_invoice_closed(state["graph_invoice_id"])
        return {"assigned_dispute_reason_code": "WRITE_OFF", "system_execution_logs": logs}
        
    # Execute verified response grounding prompting structure via the cloud infrastructure portal
    with azure_client.get_openai_client() as openai_model:
        grounding_instruction = f"""
        You are a strict financial validation checker. Determine the matching code based ONLY on the text evidence provided.
        If the text context does not explicitly specify or support a clear code condition matching the triggers, output 'UNKNOWN'.
        
        [GROUNDING EVIDENCE SNIPPET]
        "{state['retrieved_document_grounding']}"
        
        [REASON CODE TRIGGER MATRIX MAP]
        - pricing issue / charging mismatch / list variance -> D01
        - freight claim / shipping fee waiver / delivery delay -> D02
        - damage claim / broken package / crushed pallet -> D03
        - tax difference / exemption certificate -> D04
        - discount taken / early payment incentive -> D05
        
        Your response must adhere exactly to this structural constraint:
        Code: [Insert Alphanumeric Code Value here] | Justification: [Insert verbatim sentence quote from Grounding Evidence]
        """
        
        inference_pass = openai_model.chat.completions.create(
            model="gpt-5.1",
            messages=[{"role": "user", "content": grounding_instruction}]
        )
        
        grounded_output = inference_pass.choices[0].message.content.strip()
        logger.info(f"Grounded Model Response generated: {grounded_output}")
        
        # 4. Trigger write-action tool to update transaction registry tables
        create_deduction_record(state["graph_invoice_id"], state["calculated_variance_amount"], grounded_output)
        
        return {"assigned_dispute_reason_code": grounded_output, "system_execution_logs": logs}

In [8]:
# Initialize graph workflow structure map
rag_graph_builder = StateGraph(RAGAgentState)

# Append workers into the processing canvas matrix
rag_graph_builder.add_node("financial_evaluator", check_financial_ledger_node)
rag_graph_builder.add_node("rag_retrieval_processor", document_retrieval_rag_node)
rag_graph_builder.add_node("grounding_validator", response_grounding_validation_node)

# Map connecting pipelines
rag_graph_builder.add_edge(START, "financial_evaluator")
rag_graph_builder.add_edge("financial_evaluator", "rag_retrieval_processor")
rag_graph_builder.add_edge("rag_retrieval_processor", "grounding_validator")
rag_graph_builder.add_edge("grounding_validator", END)

# Compile runnable agent application framework
rag_automation_engine = rag_graph_builder.compile()
logger.info("End-to-End Stateful RAG Graph Engine successfully built, mapped, and compiled.")

2026-07-21 12:05:35,313 - [INFO] - RAGReconciliationAgent - End-to-End Stateful RAG Graph Engine successfully built, mapped, and compiled.


In [9]:
# Map test case payload parameters tracking an unmatched short payment scenario
input_payload: RAGAgentState = {
    "customer_name": "Accenture Enterprise Training Account",
    "bank_payment_value": 9500.00, # Billed amount is $10,000. Underpaid by $500.
    "remittance_search_query": "pricing charging mismatch issue details",
    "graph_invoice_id": "",
    "graph_invoice_amount": 0.0,
    "calculated_variance_amount": 0.0,
    "retrieved_document_grounding": "",
    "assigned_dispute_reason_code": "PENDING",
    "system_execution_logs": []
}

logger.info("Invoking transaction payload loop down the RAG agent state graph...")
final_output_metrics = rag_automation_engine.invoke(input_payload)

print("\n================ RAG AUTOMATION ENGINE EXECUTION TRANSACTION REPORT ================")
print(f"Target Invoice Code Identified : {final_output_metrics['graph_invoice_id']}")
print(f"Original Billed Value Amount   : ${final_output_metrics['graph_invoice_amount']:.2f}")
print(f"Identified Shortfall Variance  : ${final_output_metrics['calculated_variance_amount']:.2f}")
print(f"Isolated Grounded Document Text: '{final_output_metrics['retrieved_document_grounding']}'")
print(f"Engine Assigned Resolution Output : {final_output_metrics['assigned_dispute_reason_code']}")
print("\nSystem Component Execution Audit Trail Trace:")
for runtime_step, log_entry in enumerate(final_output_metrics["system_execution_logs"], 1):
    print(f"  [{runtime_step}] -> {log_entry}")
print("====================================================================================")

2026-07-21 12:06:03,956 - [INFO] - RAGReconciliationAgent - Invoking transaction payload loop down the RAG agent state graph...
2026-07-21 12:06:03,978 - [INFO] - RAGReconciliationAgent - --- ENTERING NODE: Financial Ledger Evaluator ---
2026-07-21 12:06:03,980 - [INFO] - RAGReconciliationAgent - [TOOL EXECUTION] -> query_open_ar looking up open balance items for: Accenture Enterprise Training Account
2026-07-21 12:06:03,981 - [INFO] - RAGReconciliationAgent - [TOOL EXECUTION] -> calculate_variance parsing math values: Billed $10000.00, Paid $9500.00
2026-07-21 12:06:03,983 - [INFO] - RAGReconciliationAgent - Math analysis result: Discrepancy evaluated to: $500.00
2026-07-21 12:06:03,991 - [INFO] - RAGReconciliationAgent - --- ENTERING NODE: Document Retrieval RAG Processor ---
2026-07-21 12:06:03,994 - [INFO] - RAGReconciliationAgent - [TOOL EXECUTION] -> search_remittance searching vector DB for context: 'pricing charging mismatch issue details'
2026-07-21 12:06:04,659 - [INFO] - RAG

In [ ]:
# --- Isolated RAG Reconciliation Dashboard App (namespaced to avoid collisions) ---
import ipywidgets as rag_widgets
from IPython.display import display as rag_display, HTML as RagHTML
import io as rag_io
import logging as rag_logging_mod

# Independent log capture stream, does not touch the existing 'logger' object
rag_log_capture_string = rag_io.StringIO()
rag_log_handler = rag_logging_mod.StreamHandler(rag_log_capture_string)
rag_log_handler.setFormatter(rag_logging_mod.Formatter("%(levelname)s - %(message)s"))

# Attach to the SAME logger used by the RAG graph nodes, without renaming or replacing it
rag_dashboard_logger = rag_logging_mod.getLogger("RAGReconciliationAgent")
rag_dashboard_logger.addHandler(rag_log_handler)

# --- Dynamic State Rendering Pipeline ---
def rag_update_dashboard_ui(payload=None, final_state=None, active_logs=""):
    """Generates and injects dynamic HTML based on active RAG runtime states."""

    cust_name = payload["customer_name"] if payload else "No Active Session"
    bank_amt = payload["bank_payment_value"] if payload else 0.0
    query_text = payload["remittance_search_query"] if payload else "Awaiting execution payload ingestion..."

    invoice_id = final_state["graph_invoice_id"] if final_state else "PENDING"
    invoice_amt = final_state["graph_invoice_amount"] if final_state else 0.0
    variance = final_state["calculated_variance_amount"] if final_state else 0.0
    grounding_text = final_state["retrieved_document_grounding"] if final_state else "Awaiting semantic retrieval..."
    reason_code = final_state["assigned_dispute_reason_code"] if final_state else "PENDING"

    log_display = active_logs if active_logs else "[SYSTEM] Ready. Configure inputs below and click 'Run RAG Reconciliation'."
    log_html_lines = log_display.replace("\n", "<br>")

    html_layout = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background-color: #f3f4f6; padding: 20px; border-radius: 8px;">
        <div style="max-width: 900px; margin: 0 auto; background: #ffffff; padding: 25px; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05);">
            <div style="border-bottom: 2px solid #e5e7eb; padding-bottom: 15px; margin-bottom: 20px;">
                <h2 style="margin: 0; color: #111827;">RAG Reconciliation Control Panel App</h2>
            </div>

            <div style="display: flex; gap: 20px; margin-bottom: 20px;">
                <div style="flex: 1; padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa;">
                    <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Customer Scope</label>
                    <div style="font-size: 15px; font-weight: 600; color: #1f2937; margin-top: 5px;">{cust_name}</div>
                </div>
                <div style="flex: 1; padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa;">
                    <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Matched Invoice</label>
                    <div style="font-size: 15px; font-weight: 600; color: #1f2937; margin-top: 5px;">{invoice_id} (${invoice_amt:,.2f})</div>
                </div>
                <div style="flex: 1; padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa;">
                    <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Calculated Variance</label>
                    <div style="margin-top: 5px;"><span style="display: inline-block; padding: 4px 10px; font-size: 12px; font-weight: bold; border-radius: 12px; background-color: #fef3c7; color: #d97706;">${variance:,.2f}</span></div>
                </div>
                <div style="flex: 1; padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa;">
                    <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Dispute Code (RAG)</label>
                    <div style="margin-top: 5px;"><span style="display: inline-block; padding: 4px 10px; font-size: 12px; font-weight: bold; border-radius: 12px; background-color: #dbeafe; color: #2563eb;">{reason_code}</span></div>
                </div>
            </div>

            <div style="padding: 15px; border: 1px solid #e5e7eb; border-radius: 6px; background: #fafafa; margin-bottom: 20px;">
                <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Search Query</label>
                <p style="margin: 6px 0 10px 0; font-style: italic; color: #374151; font-size: 13px;">"{query_text}"</p>
                <label style="font-size: 11px; color: #6b7280; text-transform: uppercase; font-weight: bold;">Retrieved Grounding Evidence</label>
                <p style="margin: 6px 0 0 0; font-style: italic; color: #374151; font-size: 13px;">"{grounding_text}"</p>
            </div>

            <label style="font-size: 11px; color: #6b7280; font-weight: bold; text-transform: uppercase;">RAG Graph Engine Reasoning Execution Logs</label>
            <div style="background: #1f2937; color: #f9fafb; font-family: monospace; padding: 15px; border-radius: 6px; height: 160px; overflow-y: auto; margin-top: 8px; font-size: 12px; line-height: 1.4;">
                {log_html_lines}
            </div>
        </div>
    </div>
    """
    rag_ui_output.clear_output(wait=True)
    with rag_ui_output:
        rag_display(RagHTML(html_layout))

# --- Widget Control Elements Initialization ---
rag_input_customer = rag_widgets.Text(value="Accenture Enterprise Training Account", description="Customer:")
rag_input_bank_amt = rag_widgets.FloatText(value=9500.00, description="Bank Paid:")
rag_input_query = rag_widgets.Text(value="pricing charging mismatch issue details", description="Search Query:", layout=rag_widgets.Layout(width='90%'))
rag_btn_execute = rag_widgets.Button(description="Run RAG Reconciliation", button_style="success", icon="play", layout=rag_widgets.Layout(width='90%', margin='10px 0px'))

rag_form_box = rag_widgets.VBox([
    rag_widgets.HTML("<h4>Configure Incoming Payment & Remittance Query</h4>"),
    rag_input_customer, rag_input_bank_amt, rag_input_query, rag_btn_execute
], layout=rag_widgets.Layout(padding="15px", border="1px solid #ccc", background_color="#fafafa", width="350px", border_radius="8px"))

rag_ui_output = rag_widgets.Output()
rag_app_container = rag_widgets.HBox([rag_form_box, rag_ui_output], layout=rag_widgets.Layout(gap="20px"))

# --- Engine Execution Trigger Connection ---
def rag_trigger_agent_execution(b):
    rag_log_capture_string.truncate(0)
    rag_log_capture_string.seek(0)

    payload_state: RAGAgentState = {
        "customer_name": rag_input_customer.value,
        "bank_payment_value": float(rag_input_bank_amt.value),
        "remittance_search_query": rag_input_query.value,
        "graph_invoice_id": "",
        "graph_invoice_amount": 0.0,
        "calculated_variance_amount": 0.0,
        "retrieved_document_grounding": "",
        "assigned_dispute_reason_code": "PENDING",
        "system_execution_logs": []
    }

    rag_dashboard_logger.info("Application layer event received. Passing state payload variables into compiled RAG graph engine...")

    try:
        runtime_final_state = rag_automation_engine.invoke(payload_state)
        captured_logs = rag_log_capture_string.getvalue()
        rag_update_dashboard_ui(payload=payload_state, final_state=runtime_final_state, active_logs=captured_logs)
    except Exception as run_err:
        rag_dashboard_logger.error(f"Execution run halted by runtime exception: {str(run_err)}")
        rag_update_dashboard_ui(payload=payload_state, final_state=None, active_logs=rag_log_capture_string.getvalue())

rag_btn_execute.on_click(rag_trigger_agent_execution)

rag_display(rag_app_container)
rag_update_dashboard_ui()

Here is the table formatted in basic Markdown for easy copying and rendering:

| Customer Input | Bank Paid ($) | Search Query Context | Matched Invoice & Billed Amount | Calculated Variance | Expected RAG Resolution / Reason Code |
| --- | --- | --- | --- | --- | --- |
| **Accenture Batch A** | 9,500.00 | pricing charging mismatch issue details | INV-2026-A1 ($10,000.00) | $500.00 | D01 (Item Pricing Mismatch) |
| **Accenture Batch B** | 9,800.00 | freight transportation shipping fee delay | INV-2026-B2 ($10,000.00) | $200.00 | D02 (Freight / Delivery Delay Waiver) |
| **Accenture Batch C** | 9,250.00 | damaged warehouse crushed pallet claim | INV-2026-C3 ($10,000.00) | $750.00 | D03 (Damaged Goods / Warehouse Claim) |
| **Accenture Batch D** | 9,100.00 | state corporate tax exemption validation | INV-2026-D4 ($10,000.00) | $900.00 | D04 (Tax Difference / Certificate Exemption) |
| **Accenture Batch E** | 9,800.00 | early payment incentive discount taken | INV-2026-E5 ($10,000.00) | $200.00 | D05 (Contractual Early Payment Discount) |
| **Accenture Batch A** | 9,997.50 | pricing charging mismatch issue details | INV-2026-A1 ($10,000.00) | $2.50 | WRITE_OFF (Auto-resolved, $\le \$5.00$ tolerance) |
| **Accenture Batch B** | 9,999.00 | freight transportation shipping fee delay | INV-2026-B2 ($10,000.00) | $1.00 | WRITE_OFF (Auto-resolved, $\le \$5.00$ tolerance) |
| **Accenture Enterprise Training Account** | 9,500.00 | short paying line item discrepancy | INV-2026-A1 ($10,000.00) | $500.00 | D01 (Semantic match to Line 3 item pricing) |
| **Accenture Batch C** | 9,500.00 | unrecognized generic deduction query | INV-2026-C3 ($10,000.00) | $500.00 | UNKNOWN (Context does not trigger matrix rules) |
| **Accenture Batch E** | 10,000.00 | early payment incentive discount taken | INV-2026-E5 ($10,000.00) | $0.00 | WRITE_OFF (Fully settled balance) |